# Interview Transcriber — launcher

Use a T4 GPU runtime and the same Google account whose Drive contains the recordings. Store the Hugging Face read-only token once as HF_TOKEN in Colab Secrets. Run the single START INTERVIEW TRANSCRIBER cell below. It mounts Drive, updates the repo, installs the tested transcription stack and an isolated Gradio UI, then launches the temporary authenticated interface.

In [ ]:
# @title ▶ START INTERVIEW TRANSCRIBER
import os
import secrets
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive, userdata

REPO_URL = "https://github.com/playply/transcriber.git"
REPO_DIR = Path("/content/transcriber")
DRIVE_MOUNT = Path("/content/drive")
UI_VENV = Path("/content/transcriber-ui-venv")

# 1) Require a GPU before doing any heavy setup.
nvidia_smi = shutil.which("nvidia-smi")
if not nvidia_smi:
    raise RuntimeError("NVIDIA GPU is not attached to this Colab runtime. Select Runtime → Change runtime type → T4 GPU, reconnect the runtime, then run START again.")
gpu_check = subprocess.run(
    [nvidia_smi],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
if gpu_check.returncode != 0:
    raise RuntimeError("NVIDIA GPU check failed. In Colab select Runtime → Change runtime type → T4 GPU, reconnect the runtime, then run START again.")
print("✓ GPU available")

# 2) Mount the same Google account whose Drive contains the recordings.
drive.mount(str(DRIVE_MOUNT))
print("✓ Google Drive mounted")

# 3) Read the Hugging Face read-only token from Colab Secrets.
try:
    hf_token = userdata.get("HF_TOKEN")
except Exception as exc:
    raise RuntimeError(
        "HF_TOKEN is not available. Add a read-only HF_TOKEN in Colab Secrets and enable notebook access."
    ) from exc
if not hf_token:
    raise RuntimeError("HF_TOKEN is empty in Colab Secrets.")
print("✓ HF_TOKEN loaded from Colab Secrets")

# 4) Get the latest application code.
if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--quiet", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/main"], check=True)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(["git", "clone", "--branch", "main", REPO_URL, str(REPO_DIR)], check=True)
print("✓ Repository ready")

# 5) Install only the tested transcription stack in the main Colab Python.
print("Installing transcription dependencies...")
core_install = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-r",
        str(REPO_DIR / "requirements.txt"),
        "python-docx",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
if core_install.returncode != 0:
    print(core_install.stdout)
    raise RuntimeError("Transcription dependency installation failed. The complete pip output is shown above.")
print("✓ Transcription dependencies installed")

# 6) Isolate Gradio from WhisperX/pyannote dependencies.
if not (UI_VENV / "bin" / "python").exists():
    subprocess.run([sys.executable, "-m", "venv", str(UI_VENV)], check=True)
ui_python = UI_VENV / "bin" / "python"

print("Installing UI dependencies...")
ui_install = subprocess.run(
    [str(ui_python), "-m", "pip", "install", "gradio==6.27.0"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
if ui_install.returncode != 0:
    print(ui_install.stdout)
    raise RuntimeError("Gradio installation failed. The complete pip output is shown above.")

ui_check = subprocess.run(
    [str(ui_python), "-c", "import gradio; print(gradio.__version__)"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    timeout=90,
)
if ui_check.returncode != 0:
    print(ui_check.stdout)
    raise RuntimeError("Gradio import check failed in the isolated UI environment.")
print("✓ UI environment ready (Gradio " + ui_check.stdout.strip() + ")")

# 7) Launch the temporary authenticated Gradio UI.
env = os.environ.copy()
env["HF_TOKEN"] = hf_token
env["TRANSCRIBER_DRIVE_ROOT"] = "/content/drive/MyDrive"
env["TRANSCRIBER_UI_PASSWORD"] = secrets.token_urlsafe(10)
env["TRANSCRIBER_APP_PYTHON"] = sys.executable
env["PYTHONUNBUFFERED"] = "1"

print("\nStarting Interview Transcriber...", flush=True)
subprocess.run([str(ui_python), "-u", str(REPO_DIR / "ui.py")], env=env, check=True)